# CS410 Project -- Phase 1
Cindy Nguyen

Prof. Michael Wilson

TA. Amber Shore

CS410 TOP: Data With Python

1/17/2026

## Project Overview
VocaDB is a fanmade wiki that contains all the information about VOCALOID, primarily songs and setlists for concerts around the world.

VOCALOID and its official concerts, along with special music events are owned and hosted by Crypton Future Media (CFM).

Although CFM has official concerts, there are also many fanmade events as well and other companies that host vocal synth concerts around the world.

VocaDB has an API which is where I'm getting my data from. I want to use this project to predict:
1. What songs will likely show up at the next VOCALOID concert in April (MIKU EXPO 2026)?
2. When will those songs play in the setlist? What is their average position?

## Importing VocaDB data from .json files
All of the data pulled from the API is stored in .json format. I created a .py script to pull all of this data from VocaDB, and formatted it a little nicer so it's easier to use/query from.

This is all of the songs from previous concert/event setlists.

In [1]:
import pandas as pd
import json

songs_df = pd.read_json("vocaloid_setlist_songs.json")
songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2274 entries, 0 to 2273
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2274 non-null   str    
 1   english_title      2274 non-null   str    
 2   artist             2274 non-null   str    
 3   length_seconds     2274 non-null   int64  
 4   setlist_frequency  2274 non-null   int64  
 5   avg_setlist_order  2274 non-null   float64
 6   song_type          2274 non-null   str    
 7   pv_services        2274 non-null   str    
 8   times_favorited    2274 non-null   int64  
 9   rating             2274 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 177.8 KB


This looks pretty great! There's no `NaN` values at all, pretty much every field is populated so this cleaning step isn't necessary.

## Data Cleansing
Even if there are no `NaN` values, we still have to clean the data. And given that this is numerical data, cleaning it is sort of a different process.

### Computing the average setlist order (avg_setlist_order)
One of the data cleansing steps actually has to be done outside of Jupyter notebook since it has to do with how I retrieved my data and computed the average setlist order when creating the dataset itself.

VOCALOID is a big community of fans, full of very talented producers, musicians, and also DJs. This doesn't sound terrible until you realize that these special DJ events can go on for a really long time and mashup tons of songs together.

This is how long the setlist is for the MIKU EXPO Digital Stars 2021 Online event...

93 songs is a LOT. Fortunately, "MONSTER" was only ever played once at this special event so it's not going to skew the data for other instances of it being played.

However, there were other songs that are much more popular that were in this setlist as well (Melt, Glass Wall, to name a few). These songs would typically show up much earlier in the setlist across multiple regular concerts, but are played much later at DJ events to build up excitement for the audience. 

This is a huge problem, since this will compute the average order to be much higher than it's supposed to be.

It's hard to demonstrate here, but what I'm going to do is exclude any setlists with > 50 songs from my dataset by modifying my Python script.

__**I can do this relatively safely for a few reasons**__:
1. From my own personal knowledge, most concerts average to around ~30 songs total
    - There are a couple exceptions:
       - Kikuoland (49)
       - Project SEKAI COLORFUL LIVE 4th - Unison (48)
       - VVV MUSIC LIVE concert series (~56)
2. Choosing the number "50" will cover the two main exceptions above, but exclude any super long special events (i.e. DJ sets) which are the MOST likely to skew the value of the average order negatively
3. It's safe to not include any of the VVV MUSIC LIVE concerts (or care at all about VVV MUSIC LIVE)
   - These concerts use voice/character models that would never show up at a MIKU EXPO concert due to licensing reasons UNLESS a major breakthrough happens where a collaboration is viable
      - Nowadays, this is incredibly rare, and in modern times has only happened ONCE with 32ki's song "Mesmerizer" where Kasane Teto made a guest appearance for the JAPAN LIVE TOUR 2025 ~ BLOOMING ~ (and this isn't even a main concert event!)
   - Even if any of the performers had a chance of appearing at MIKU EXPO (e.g. GUMI since she is partially licensed under CFM still), the songs from VVV MUSIC LIVE would never be performed at MIKU EXPO due to licensing reasons

### Before/After Results of Removing and Recalculating

#### Here is the dataset *before* removing all setlists longer than 50 tracks, along with the average positions computed including those long setlists.

In [2]:
songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2274 entries, 0 to 2273
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2274 non-null   str    
 1   english_title      2274 non-null   str    
 2   artist             2274 non-null   str    
 3   length_seconds     2274 non-null   int64  
 4   setlist_frequency  2274 non-null   int64  
 5   avg_setlist_order  2274 non-null   float64
 6   song_type          2274 non-null   str    
 7   pv_services        2274 non-null   str    
 8   times_favorited    2274 non-null   int64  
 9   rating             2274 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 177.8 KB


In [3]:
# Setting this to 30 so we can actually see the avg_setlist_order be > 1

mask = songs_df["setlist_frequency"] > 30

frequent_songs = songs_df[mask]

frequent_songs

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,37,14.270270,Original,"NicoNicoDouga, Youtube",388,1858
73,ルカルカ★ナイトフィーバー,Luka Luka★Night Fever,samfree feat. 巡音ルカ,239,39,16.307692,Original,"NicoNicoDouga, Youtube",261,1329
83,メルト,Melt,ryo feat. 初音ミク,256,41,24.292683,Original,"NicoNicoDouga, Youtube, SoundCloud",297,1522
149,リモコン,Remote Control,"じーざすP, WONDERFUL★OPPORTUNITY! feat. 鏡音リン, 鏡音レン",312,31,14.419355,Original,"NicoNicoDouga, Youtube",206,1086
152,ワールドイズマイン,World is Mine,"ryo, supercell feat. 初音ミク",255,62,13.500000,Original,"NicoNicoDouga, Youtube",303,1609
188,???,None,,0,75,19.946667,Other,Nothing,2,12
268,Tell Your World,None,kz feat. 初音ミク,257,66,19.045455,Remix,Youtube,218,1226


#### And here is the new dataset *after* removing setlists with more than 50 songs and recalculating the average positions.

In [4]:
clean_songs_df = pd.read_json("CLEAN_vocaloid_setlist_songs.json")

clean_songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2227 entries, 0 to 2226
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2227 non-null   str    
 1   english_title      2227 non-null   str    
 2   artist             2227 non-null   str    
 3   length_seconds     2227 non-null   int64  
 4   setlist_frequency  2227 non-null   int64  
 5   avg_setlist_order  2227 non-null   float64
 6   song_type          2227 non-null   str    
 7   pv_services        2227 non-null   str    
 8   times_favorited    2227 non-null   int64  
 9   rating             2227 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 174.1 KB


In [5]:
# Setting this to 30 so we can actually see the avg_setlist_order be > 1

mask = clean_songs_df["setlist_frequency"] > 30

clean_frequent_songs = clean_songs_df[mask]

clean_frequent_songs

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,36,12.361111,Original,"NicoNicoDouga, Youtube",388,1858
73,ルカルカ★ナイトフィーバー,Luka Luka★Night Fever,samfree feat. 巡音ルカ,239,39,16.307692,Original,"NicoNicoDouga, Youtube",261,1329
83,メルト,Melt,ryo feat. 初音ミク,256,41,24.292683,Original,"NicoNicoDouga, Youtube, SoundCloud",297,1522
149,リモコン,Remote Control,"じーざすP, WONDERFUL★OPPORTUNITY! feat. 鏡音リン, 鏡音レン",312,31,14.419355,Original,"NicoNicoDouga, Youtube",206,1086
152,ワールドイズマイン,World is Mine,"ryo, supercell feat. 初音ミク",255,62,13.500000,Original,"NicoNicoDouga, Youtube",303,1609
188,???,None,,0,65,13.784615,Other,Nothing,2,12
268,Tell Your World,None,kz feat. 初音ミク,257,65,18.169231,Remix,Youtube,218,1226


#### Results
If you're like me and couldn't tell the difference at first, here's a specific example using "Senbonzakura":

In [6]:
frequent_songs.head(1)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,37,14.27027,Original,"NicoNicoDouga, Youtube",388,1858


In [7]:
clean_frequent_songs.head(1)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,36,12.361111,Original,"NicoNicoDouga, Youtube",388,1858


The average setlist position changed! While that's only 2 spots earlier in the setlist for this particular song, that's still easily about 10 minutes of your life waiting for a fan favorite title to play.

2 spots isn't that big of a deal, but for other songs that are even more popular and were included in larger DJ setlists, the changes in the average setlist position could've been much more dramatic.

### Songs with 0 seconds in length

So you might've noticed something funky from the output earlier, and it's that there are songs that are 0 seconds in duration.

In [8]:
zero_length_songs = clean_songs_df["length_seconds"] == 0
clean_songs_df[zero_length_songs]

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
0,STARLIGHT,None,失いP feat. 花隈千冬,0,1,1.0,Cover,Nothing,0,0
5,物語の始まり,Monogatari no Hajimari,失いP feat. 花隈千冬,0,1,6.0,Cover,Nothing,0,0
9,ビーチストーリー,Beach Story,失いP feat. 京町セイカ AI,0,1,10.0,Cover,Nothing,0,0
10,黒いメガネ,Kuroi Megane,失いP feat. 花隈千冬,0,1,11.0,Cover,Nothing,0,0
11,僕の命,Boku no Inochi,失いP feat. 桜乃そら SV,0,1,12.0,Cover,Nothing,0,0
...,...,...,...,...,...,...,...,...,...,...
2147,地球毁灭的那一刻,When the doomsday comes,"KIDE, 动点P feat. 星尘",0,1,5.0,Original,Nothing,2,10
2182,All About That Bass,None,Dharma feat. 京町セイカ (Unknown),0,1,4.0,Cover,Nothing,1,3
2183,Feel Special,None,ぎゅや feat. 京町セイカ (Unknown),0,1,5.0,Cover,Nothing,0,0
2187,天城越え,Walk Over Amagi Pass,ぎゅや feat. 京町セイカ (Unknown),0,1,11.0,Cover,Nothing,0,0


This information is hidden and won't show up on the website, but according to VocaDB, one of these songs is *supposed* to be "FE!N" by Travis Scott.

As funny as this is, no, CFM has never collaborated with Travis Scott and FE!N is definitely not a Japanese song.

I mentioned it earlier, but VocaDB is a fanmade wiki. And the "fanmade" parts of it really shine through in moments like this.

All humans make mistakes. Some of the mistakes can be incredibly funny (and intentional). Maybe someone's engaging in internet trolling to make my life more inconvenient. Though sometimes, human error and random things *just* happen. The other song titles seem very legitimate - and they are.

__**Theories as to why `0` second data happens**__:
- A fan is entering in bad data because they think it's funny (it kind of is (it's really inconvenient though))
- Someone entered the data incorrectly but forgot to remove the information from the database itself
- Some of the songs had actual durations but disappeared over time
   - "Monogatari no Hajimari" is supposed to be 4:27 (`267` seconds), but shows up as `0` because the original link to the song doesn't exist anymore (the artist deleted the song or the link to that song vanished for whatever reason)
- Artists delete their old works all the time, and it becomes lost media very quickly in the VOCALOID community to the point that there's a subcommunity that's dedicated to finding the origin of lost songs. If no one archives it, there's no trace of it existing other than basic information that might've been input into the database before it was deleted

Whatever the case may be, we don't really care. I care because I'm a fanatic, but for data processing, cleaning, and the sake of this project, we don't want this data! We're going to get rid of it!

In [9]:
# Drop all songs where their length in seconds is 0
# Only keep songs where their length is greater than 0
clean_songs_df = clean_songs_df.loc[clean_songs_df['length_seconds'] > 0]

clean_songs_df

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
1,Love is in Trouble -恋愛主義バージョン-,None,失いP feat. 弦巻マキ AI (Japanese),236,1,2.0,Cover,Youtube,3,13
2,HEADSTRONG -恋愛主義バージョン-,HEADSTRONG -Renai Shugi Version--,失いP feat. 京町セイカ AI,241,1,3.0,Cover,Youtube,1,5
3,精神疾患 -恋愛主義バージョン-,Mental Illness -Love Doctrine Version-,失いP feat. 夏色花梨,248,1,4.0,Cover,Youtube,4,14
4,愛をつかむ -恋愛主義バージョン-,Ai wo Tsukamu -Renai Shugi Version-,失いP feat. 小春六花 AI,311,1,5.0,Cover,Youtube,4,14
6,CAFETERIA -恋愛主義バージョン-,None,失いP feat. ついなちゃん AI,261,1,7.0,Cover,Youtube,3,11
...,...,...,...,...,...,...,...,...,...,...
2222,31 secrets,None,毎夜P feat. 重音テト,245,1,15.0,Original,Youtube,8,42
2223,ウルトラトレーラー,Ultra Trailer,マサラダ feat. 重音テトSV,272,1,16.0,Original,"NicoNicoDouga, Youtube, Bilibili",70,328
2224,吉原ラメント-再来盤-,Yoshiwara Lament -Re-recording-,亜沙 feat. 重音テトSV,224,1,17.0,Remaster,"NicoNicoDouga, Youtube",22,132
2225,優しくなれたら,Yasashikunaretara,亜沙 feat. 重音テトSV,202,1,18.0,Original,Youtube,2,12


In [10]:
clean_songs_df.info()

<class 'pandas.DataFrame'>
Index: 2099 entries, 1 to 2226
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2099 non-null   str    
 1   english_title      2099 non-null   str    
 2   artist             2099 non-null   str    
 3   length_seconds     2099 non-null   int64  
 4   setlist_frequency  2099 non-null   int64  
 5   avg_setlist_order  2099 non-null   float64
 6   song_type          2099 non-null   str    
 7   pv_services        2099 non-null   str    
 8   times_favorited    2099 non-null   int64  
 9   rating             2099 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 180.4 KB


In [11]:
clean_songs_df.describe()

,length_seconds,setlist_frequency,avg_setlist_order,times_favorited,rating
count,2099.000000,2099.000000,2099.000000,2099.000000,2099.000000
mean,234.677465,2.441639,13.449130,29.928537,163.821820
std,79.100950,3.966350,9.764709,55.674499,280.739203
min,41.000000,1.000000,1.000000,0.000000,0.000000
25%,202.000000,1.000000,6.000000,2.000000,12.000000
50%,231.000000,1.000000,11.227273,8.000000,46.000000
75%,263.000000,2.000000,18.000000,30.000000,188.500000
max,2304.000000,65.000000,50.000000,648.000000,2985.000000


In [12]:
clean_songs_df.query('length_seconds == 41')

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
2048,またね,Till We Meet Again,"ryo, supercell feat. 初音ミク",41,1,41.0,Remaster,Youtube,11,55


We went from 2227 songs to 2099 songs, removing all 128 songs that were 0 seconds. Now we can see the shortest song is 41 seconds, which makes a lot more sense than a song that practically doesn't exist.

"またね" by ryo is meant to be an outro song and was played at the end of a concert. Concerts used to have outro songs like this one, but it's more likely nowadays that they play the instrumental of the concert theme to close out for the evening which isn't as easy to register into the database without causing duplicate data.

### Songs with no PV services -- Justifications

Some songs have been played at concerts but don't have any PV services. This means that it doesn't have a link to an original source (YouTube, SoundCloud, Spotify, Piapro, NicoNicoDouga, Bilibili, etc.)

There are quite a few of these actually.

In [13]:
no_pvs_list = clean_songs_df.query('pv_services == "Nothing"')
no_pvs_list.info()

<class 'pandas.DataFrame'>
Index: 49 entries, 205 to 2083
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              49 non-null     str    
 1   english_title      49 non-null     str    
 2   artist             49 non-null     str    
 3   length_seconds     49 non-null     int64  
 4   setlist_frequency  49 non-null     int64  
 5   avg_setlist_order  49 non-null     float64
 6   song_type          49 non-null     str    
 7   pv_services        49 non-null     str    
 8   times_favorited    49 non-null     int64  
 9   rating             49 non-null     int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 4.2 KB


In [14]:
no_pvs_list.head(10)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
205,风兮风兮,None,ilem feat. 洛天依 V4 (萌),216,1,8.0,Original,Nothing,2,10
250,Berlin Calling,None,"EHAMIC feat. 結月ゆかり, 石黒千尋",176,1,9.0,Cover,Nothing,0,0
251,月旅,Tsuki Tabi,GYARI feat. 結月ゆかり,282,1,14.0,Original,Nothing,2,10
294,香草茶与黑咖啡,Vanilla Tea and Black Coffee,"苏逸_Suyi, 动点P feat. 洛天依, 洛天依 V4 日本語 (Sweet)",202,1,2.0,Original,Nothing,1,7
382,セカイ,WORLD,Jordan Forester feat. 蒼姫ラピス,258,1,1.0,Cover,Nothing,0,0
383,ビーバー,None,Jordan Forester feat. 蒼姫ラピス,234,1,2.0,Cover,Nothing,0,0
384,ツギハギスタッカート,Patchwork Staccato,Jordan Forester feat. 蒼姫ラピス,251,1,3.0,Cover,Nothing,0,0
385,ジッタードール,Jitter Doll,Jordan Forester feat. 蒼姫ラピス,240,1,4.0,Cover,Nothing,0,0
386,p.h.,None,Jordan Forester feat. 蒼姫ラピス,156,1,5.0,Cover,Nothing,0,0
387,シャルル,Charles,Jordan Forester feat. 蒼姫ラピス,228,1,6.0,Cover,Nothing,0,2


A lot of these songs are covers that we can remove safely. However, there are a few originals, remixes, or remasters in there which we definitely want to keep.

__**We want to remove these covers that have no PV, but not *ALL* covers from the database for a few reasons**__:
1. It's possible that a popular VOCALOID song is *actually* a cover of an originally human song, but fans are simply unaware of it
   - If there is a song registered like this in the database, it would not be a good idea to remove a popular song simply because it's a cover. We want to see all possible songs that will show up, regardless of whether or not it's an original
2. Inspecting closely at the song titles, a vast majority of these songs without any PV are covers that could not be played at an official event due to licensing reasons
   - The list is relatively large (which is why I only display the first 10 entries), but one of the songs is "IDOL" which is originally produced by YOASOBI for the anime "Oshi no Ko". This would definitely NOT show up at a CFM-licensed concert
   - A vast majority of these song covers come from fan events where copyright/licensing isn't as strict, hence why it shows up here
3. You might notice a lot of the top entries are made by someone named "Jordan Forester" using "Aoki Lapis" as the voicebank. While all of their covers are fully VOCALOID songs that didn't originally have a human singer, fans still attribute a particular VOCALOID to be the "original singer"
   - "Patchwork Staccato" is originally sung by Hatsune Miku and has been performed multiple times across different concerts. However, only Miku herself would ever sing that song at an official event. Aoki Lapis would never show up to perform this track since it's not "her song" and the community does not associate her with it
   - A notable exception to this rule would be the Project Sekai concerts, where human-voiced game characters appear on stage and perform *with* the virtual singer. But these tracks are registered separately as a cover and have been performed very sparsely. These covers also have PV services because they're officially licensed by CFM, so they won't show up here

#### Other songs with no PVs

There's still the fact that while most of the no-PV songs are covers, that doesn't mean that all of them are.

There are a few other categories that show up:
- Original
- Remaster
- MusicPV
- Remix

In [15]:
no_pvs_list.query('song_type != "Cover"')

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
205,风兮风兮,None,ilem feat. 洛天依 V4 (萌),216,1,8.000000,Original,Nothing,2,10
251,月旅,Tsuki Tabi,GYARI feat. 結月ゆかり,282,1,14.000000,Original,Nothing,2,10
294,香草茶与黑咖啡,Vanilla Tea and Black Coffee,"苏逸_Suyi, 动点P feat. 洛天依, 洛天依 V4 日本語 (Sweet)",202,1,2.000000,Original,Nothing,1,7
469,シリョクケンサ,Eye Examination,40mP feat. GUMI,267,1,17.000000,Remaster,Nothing,2,8
477,Cross Road,None,19's Sound Factory feat. 初音ミク,238,1,6.000000,Original,Nothing,1,7
627,Fire◎Flower 2011,None,halyosy feat. 鏡音レン,262,1,21.000000,Remaster,Nothing,7,35
875,8102,None,COP feat. 洛天依 V4 (凝),276,1,9.000000,Original,Nothing,1,3
895,"Nostalogic [""M-E-I-K-O"" mix]",None,yuukiss feat. MEIKO,271,3,12.666667,Remaster,Nothing,4,18
1357,Tornado,None,Various artists,173,2,7.000000,MusicPV,Nothing,2,6
1654,Witness (Vocamerica Mix),Witness (Vocamerica Mix),EmpathP feat. DAINA,208,1,11.000000,Remix,Nothing,1,3


Rather than delete them anyway, my justification to remove or keep some of these tracks are a little bit different than the covers that have no PV services, and some of these cases are actually pretty odd that it's worth explaining what it is.

__**Extra notes before removal**__:

The lack of PV services implies that these songs are unreachable, undistributed anywhere else officially, and potentially unusable in future events. This may sound like a case of "remove everything", but I'll point out specific entries that I'm going to keep/remove.
1. Originals
   - __Original tracks I'm going to leave alone__ on the basis that it may appear in a concert again in the future even if the likelihood is low
      - It's more likely that these tracks will show up at a fan concert, and while I'm not necessarily interested in fan setlists, I am interested in predicting what may show up at the official MIKU EXPO. Fan concerts undeniably demonstrate what fans are interested in, and also show what trends and songs are popular at the time. CFM definitely accounts for fan behavior/fan favorites when creating events, so it's important to include
2. Remasters
   - __I'll be removing all remasters__ from this list **EXCEPT** for "Nostalogic"
      - "Fire◎Flower" is a popular song. However, "Fire◎Flower 2011" has only been played once. This is likely a remastered version produced privately just for that particular concert, that the chance of this audio track ever being reused is low
      - The same also applies to "Eye Examination" - a remastered version was made but never released officially
      - "Nostalogic ["M-E-I-K-O" mix]" I will keep for a couple of reasons:
         1. It has been played more than once
         2. There are multiple different versions of "Nostalogic" - various covers, remasters, remixes, and remasters of those remixes. It's an old and very iconic song for MEIKO, so there are many versions of it that have been redone over the years and the old versions of the song are still played at events. It's best not to remove it 
3. Remixes:
      - __I'll be removing all remixes__
      - "Witness (Vocamerica Mix)" is concert-specific to the VOCAMERICA fan concert series. While the original song is available and has PV services, this remix was only for a particular event in the series that it was never officially released
      - All of the remixes by "Colorful Palette feat. various" were for one of the Project Sekai live virtual concerts - in this particular case, they were for the "RESONANCE BEATS" event
         - These tracks are pretty interesting - there's not much information out there about it, but based on the extra notes for "DAYBREAK FRONTLINE" on the wiki, it seems like these remixes were just for this collaboration event between Leo/need and Vivid BAD SQUAD (two in-game music groups) and the remixes were never meant to be released officially anywhere after the event was over
4. MusicPVs
   - __I will remove "Tornado" from the list__
      - Trying to find what this is was kind of complicated. Essentially, it doesn't exist anymore so it should be removed
         - Full explanation: The PV doesn't exist on YouTube, but the song does (not on YouTube - it exists on NicoNicoDouga), but the song used in the PV was just a cover using Megurine Luka. The original song is better known as "Tatsumaki", and the original uses Hatsune Miku as the vocalist. However, the original Miku version was deleted, and the only one that exists now is the version that uses SF-A2 Miki

__**Extra information about China-specific tracks (and why I'm not removing them)**__:
   - "风兮风兮" ["The Wind The Wind"] is performed by Luo Tianyi, who is a Chinese virtual singer licensed by CFM. While this song may be popular in China and might play at a lot of her China-specific concerts, we have no easy way of finding this information and confirming
   - Due to China's strict internet rules and the fact that Bilibili is not easily accessible outside of China, this information often never gets reuploaded or discussed on the western side of the fandom
   - While it's unlikely that Luo Tianyi or other Chinese virtual idols would perform in America, because they are officially licensed by CFM who hosts MIKU EXPO, there is technically always a non-zero chance that any of them could appear at an event
      - Hatsune Miku and Luo Tianyi have actually performed together multiple times in the past in China, and given that they both have a great relationship with the Chinese fans, it wouldn't be surprising if Luo Tianyi made a surprise appearance at some point in America
   


### Songs with no PV Services -- ACTUAL REMOVAL STEPS (pt 1.)

I spent an entire section justifying what should and shouldn't be removed. I'm fully aware it's incredibly long, and I'm sorry. Anyways, now I'll actually clean up the data.

To recap an incredibly long section...

__**We want to remove**__:
- All covers with no PVs
- All remixes with no PVs
- "Fire◎Flower 2011"
- "シリョクケンサ"/"Eye Examination"
- "Tornado" Music PV

__**We want to keep**__:
- Original tracks
- "Nostalogic ["M-E-I-K-O" mix]"
- Chinese-specific tracks (covered by keeping all originals)

In [16]:
# I'm going to create a temporary dataframe that holds all the entries I want to remove
to_remove_no_pv_items = no_pvs_list.copy()

### Excluding from the list entries we want to keep ### 
# Exclude all originals as we want to keep these
to_remove_no_pv_items = to_remove_no_pv_items.query('song_type != "Original"')

# We want to exclude "Nostalogic ["M-E-I-K-O" mix]"
to_remove_no_pv_items = to_remove_no_pv_items.query('title != \'Nostalogic ["M-E-I-K-O" mix]\'')

#### Here is our list of items that we want to remove:

In [17]:
# Display the list just to confirm we're not going to remove the wrong information

to_remove_no_pv_items.head(5)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
250,Berlin Calling,None,"EHAMIC feat. 結月ゆかり, 石黒千尋",176,1,9.0,Cover,Nothing,0,0
382,セカイ,WORLD,Jordan Forester feat. 蒼姫ラピス,258,1,1.0,Cover,Nothing,0,0
383,ビーバー,None,Jordan Forester feat. 蒼姫ラピス,234,1,2.0,Cover,Nothing,0,0
384,ツギハギスタッカート,Patchwork Staccato,Jordan Forester feat. 蒼姫ラピス,251,1,3.0,Cover,Nothing,0,0
385,ジッタードール,Jitter Doll,Jordan Forester feat. 蒼姫ラピス,240,1,4.0,Cover,Nothing,0,0


In [18]:
# And also show that it's not JUST covers in here - remasters, remixes, and music PVs are also included in here to be removed
to_remove_no_pv_items.query('song_type != "Cover"')

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
469,シリョクケンサ,Eye Examination,40mP feat. GUMI,267,1,17.0,Remaster,Nothing,2,8
627,Fire◎Flower 2011,None,halyosy feat. 鏡音レン,262,1,21.0,Remaster,Nothing,7,35
1357,Tornado,None,Various artists,173,2,7.0,MusicPV,Nothing,2,6
1654,Witness (Vocamerica Mix),Witness (Vocamerica Mix),EmpathP feat. DAINA,208,1,11.0,Remix,Nothing,1,3
1876,Blessing,None,Colorful Palette feat. various,277,1,10.0,Remix,Nothing,0,2
1882,ヒバナ -Reloaded-,HIBANA -Reloaded-,Colorful Palette feat. various,204,1,9.0,Remix,Nothing,1,5
1883,DAYBREAK FRONTLINE,None,Colorful Palette feat. various,205,1,10.0,Remix,Nothing,2,8
1884,NEO,None,Colorful Palette feat. various,185,1,11.0,Remix,Nothing,1,3


In [19]:
# There are 39 songs that we want to remove from our actual dataframe
to_remove_no_pv_items.info()

<class 'pandas.DataFrame'>
Index: 39 entries, 250 to 1884
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              39 non-null     str    
 1   english_title      39 non-null     str    
 2   artist             39 non-null     str    
 3   length_seconds     39 non-null     int64  
 4   setlist_frequency  39 non-null     int64  
 5   avg_setlist_order  39 non-null     float64
 6   song_type          39 non-null     str    
 7   pv_services        39 non-null     str    
 8   times_favorited    39 non-null     int64  
 9   rating             39 non-null     int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 3.4 KB


#### Songs with no PV Services -- ACTUAL REMOVAL STEPS (pt. 2)
Now that we have our list of items to remove, we need to remove it from the actual database entirely.

In [20]:
# Check to see what songs are on the "no fly" list in our main dataframe, store it in a new dataframe
condition = clean_songs_df.isin(to_remove_no_pv_items)

# Anything with all True values is a perfect match to the entries we want to remove and should be obliterated on sight
#condition

# Create a new dataframe - this should hold ALL songs but exclude the 39 items on the remove list
all_clean_songs_df = clean_songs_df.drop(to_remove_no_pv_items[condition].index)

all_clean_songs_df.count()

title                2060
english_title        2060
artist               2060
length_seconds       2060
setlist_frequency    2060
avg_setlist_order    2060
song_type            2060
pv_services          2060
times_favorited      2060
rating               2060
dtype: int64

In [21]:
# We can compare the count to the original dataframe we were working with and confirm that the 39 entries are gone
clean_songs_df.count()

title                2099
english_title        2099
artist               2099
length_seconds       2099
setlist_frequency    2099
avg_setlist_order    2099
song_type            2099
pv_services          2099
times_favorited      2099
rating               2099
dtype: int64

In [22]:
# Another way to confirm is to check if any of the removal list items are in the new dataframe at all using the isin() method
# The entire dataframe is False so I'm going to use head() to truncate the output
to_remove_no_pv_items.isin(all_clean_songs_df).head()

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
250,False,False,False,False,False,False,False,False,False,False
382,False,False,False,False,False,False,False,False,False,False
383,False,False,False,False,False,False,False,False,False,False
384,False,False,False,False,False,False,False,False,False,False
385,False,False,False,False,False,False,False,False,False,False


## Final Results
Between all of the data cleansing, justifications, and mini VOCALOID history + culture lessons, here is our final dataset.

__**Recap on Data Cleanse**__:
- Fixed the average setlist order being calculated with bad data
   - Readjusted my Python script to not include setlists larger than 50 as to not corrupt the dataset
   - Recalculated the average setlist order after excluding these setlists so the average turns out the way we expect it to 
- Removed all songs `0` seconds in length
- Removed all covers + remixes with no PV services, while keeping some entries due to circumstance/history

In [23]:
# Print the dataset one last time to demonstrate the results

all_clean_songs_df

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
1,Love is in Trouble -恋愛主義バージョン-,None,失いP feat. 弦巻マキ AI (Japanese),236,1,2.0,Cover,Youtube,3,13
2,HEADSTRONG -恋愛主義バージョン-,HEADSTRONG -Renai Shugi Version--,失いP feat. 京町セイカ AI,241,1,3.0,Cover,Youtube,1,5
3,精神疾患 -恋愛主義バージョン-,Mental Illness -Love Doctrine Version-,失いP feat. 夏色花梨,248,1,4.0,Cover,Youtube,4,14
4,愛をつかむ -恋愛主義バージョン-,Ai wo Tsukamu -Renai Shugi Version-,失いP feat. 小春六花 AI,311,1,5.0,Cover,Youtube,4,14
6,CAFETERIA -恋愛主義バージョン-,None,失いP feat. ついなちゃん AI,261,1,7.0,Cover,Youtube,3,11
...,...,...,...,...,...,...,...,...,...,...
2222,31 secrets,None,毎夜P feat. 重音テト,245,1,15.0,Original,Youtube,8,42
2223,ウルトラトレーラー,Ultra Trailer,マサラダ feat. 重音テトSV,272,1,16.0,Original,"NicoNicoDouga, Youtube, Bilibili",70,328
2224,吉原ラメント-再来盤-,Yoshiwara Lament -Re-recording-,亜沙 feat. 重音テトSV,224,1,17.0,Remaster,"NicoNicoDouga, Youtube",22,132
2225,優しくなれたら,Yasashikunaretara,亜沙 feat. 重音テトSV,202,1,18.0,Original,Youtube,2,12


<p style="text-align: center; font-size: 50px;" >Thank you!</p>



<p style="text-align: center; font-size: 12px;" >I'm tired... time for a break...</p>